In [24]:
import pandas as pd
import os
import numpy as np
from datetime import datetime, timedelta
import re

In [2]:
pd.set_option('display.max_columns', None)

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
RANDOM_SEED = 42
N_MATCH     = 3   # normals per cancer patient

#### <span style="color:#FF6347;">**READ**</span> 

In [32]:
# file_path = "P:/Dataset/MO-DBT-data-curation/Cancer/mo_cancer.xlsx"
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "mo_cancer" + ".xlsx")
mo_cancer = pd.read_excel(file_path)
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "normal_candidate" + ".xlsx")
normal_candidate = pd.read_excel(file_path)

file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "mo_cancer_ehr_@_index_1" + ".xlsx")
mo_cancer_ehr = pd.read_excel(file_path)
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "normal_candidate_ehr_@_index_1" + ".xlsx")
normal_candidate_ehr = pd.read_excel(file_path)

# **Step 1. BI-RADS**

In [22]:
# ── STEP 1: Bilateral BIRADS combo at INDEX-1 ─────────────────────────────────
SHORT = {'1 - Negative': '1', '2 - Benign finding': '2'}

c1 = mo_cancer[mo_cancer['Index'] == 'INDEX-1'].copy()
n1 = normal_candidate[normal_candidate['Index'] == 'INDEX-1'].copy()

def get_bilateral_combo(df):
    df = df.copy()
    df['CAT'] = df['FINDING_CATEGORY'].map(SHORT)

    side_dedup = df.drop_duplicates(subset=['PATIENT_STUDY_ID', 'SIDE'], keep='first')

    valid = side_dedup[side_dedup['SIDE'].isin(['L', 'R'])]
    piv = valid.pivot_table(
        index='PATIENT_STUDY_ID', columns='SIDE', values='CAT', aggfunc='first'
    )
    piv.columns.name = None
    def combo(row):
        vals = sorted([v for v in row.dropna()])
        return '+'.join(vals) if vals else None
    piv['birads_combo'] = piv.apply(combo, axis=1)
    result = piv[['birads_combo']].reset_index()

    nan_side_pids = set(df['PATIENT_STUDY_ID']) - set(result['PATIENT_STUDY_ID'])
    if nan_side_pids:
        nan_side = (
            df[df['PATIENT_STUDY_ID'].isin(nan_side_pids)]
            .drop_duplicates(subset=['PATIENT_STUDY_ID'], keep='first')
            [['PATIENT_STUDY_ID', 'CAT']]
            .rename(columns={'CAT': 'birads_combo'})
        )
        result = pd.concat([result, nan_side], ignore_index=True)

    return result

c_combo = get_bilateral_combo(c1)
n_combo = get_bilateral_combo(n1)

# ── Distribution table ────────────────────────────────────────────────────────
label_map = {
    '2+2': '2+2 (both Benign)',
    '1+1': '1+1 (both Negative)',
    '1+2': '1+2 (mixed)',
    '2':   '2 only (unilateral) + N/A',
    '1':   '1 only (unilateral) + N/A',
}

def combo_table(df, label):
    n_total = df['PATIENT_STUDY_ID'].nunique()
    counts  = df['birads_combo'].value_counts(dropna=False)
    tbl = pd.DataFrame({
        'Combo': counts.index.map(lambda x: label_map.get(str(x), str(x))),
        'Count': counts.values,
        '%':     (counts.values / n_total * 100).round(1),
    })
    print(f"{'─'*45}")
    print(f"{label}  (n={n_total:,})")
    print(f"{'─'*45}")
    print(tbl.to_string(index=False))
    print()

combo_table(c_combo, 'MO Cancer')
combo_table(n_combo, 'Normal Candidate')

─────────────────────────────────────────────
MO Cancer  (n=253)
─────────────────────────────────────────────
                    Combo  Count    %
        2+2 (both Benign)    197 77.9
      1+1 (both Negative)     51 20.2
2 only (unilateral) + N/A      5  2.0

─────────────────────────────────────────────
Normal Candidate  (n=30,903)
─────────────────────────────────────────────
                    Combo  Count    %
        2+2 (both Benign)  17916 58.0
      1+1 (both Negative)   9462 30.6
1 only (unilateral) + N/A   3277 10.6
2 only (unilateral) + N/A    248  0.8



# **Step 2. Race**

In [33]:
c_race_cols = [c for c in mo_cancer_ehr.columns if c.startswith('RACE_')]
n_race_cols = [c for c in normal_candidate_ehr.columns if c.startswith('RACE_')]

In [34]:
ASIAN_GROUPS = {
    'KOREAN', 'OTHER ASIAN', 'VIETNAMESE', 'CHINESE', 'FILIPINO',
    'JAPANESE', 'INDIAN (ASIAN)'
}

PACIFIC_GROUPS = {
    'HAWAIIAN',  'OTHER PACIFIC ISLANDER'
}

def get_simplified_race(df):
    race_cols = [c for c in df.columns if c.startswith('RACE_')]
    df2 = df[race_cols].fillna(0)
    all_zero = df2.sum(axis=1) == 0
    race = df2.idxmax(axis=1).str.replace('RACE_', '')
    race[all_zero] = 'UNKNOWN'
    def simplify(r):
        if r == 'WHITE':  return 'WHITE'
        if r == 'BLACK':  return 'BLACK'
        if r == 'AMERICAN INDIAN/ALASKA NATIVE':  return 'AMERICAN INDIAN/ALASKA NATIVE'
        if r in PACIFIC_GROUPS: return 'HAWAIIAN/PACIFIC ISLANDER'
        if r in ASIAN_GROUPS: return 'ASIAN'
        return 'OTHER/UNKNOWN'
    return race.apply(simplify)

mo_cancer_ehr['race']        = get_simplified_race(mo_cancer_ehr)
normal_candidate_ehr['race'] = get_simplified_race(normal_candidate_ehr)

In [38]:
mo_cancer_ehr

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,EXAM_COMPLETED_DATE,Study,Index,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,RISK_FACTOR_No family history of breast cancer,RISK_FACTOR_Personal breast cancer history,"RISK_FACTOR_Weak family history of breast cancer (aunt, grandmother, cousin)",RISK_FACTOR_Late child bearing (after 30),"RISK_FACTOR_Very strong family history of breast cancer (mother, sister, daughter, pre-menopause or multiple post-menopausal first degree relatives)","RISK_FACTOR_Family history of ovarian cancer in mother, sister, or daughter",RISK_FACTOR_Post-menopausal patient,RISK_FACTOR_BRCA1 gene mutation,RISK_FACTOR_Nulliparous,RISK_FACTOR_Family history of ovarian cancer in distant blood relatives,"RISK_FACTOR_Intermediate family history of breast cancer (mother, sister, daughter, post-menopause)","RISK_FACTOR_History of high risk lesion on previous biopsy, LCIS, atypical hyperplasia",RISK_FACTOR_Previous chest radiation therapy,RISK_FACTOR_Other family history of breast cancer,RISK_FACTOR_BRCA2 gene mutation,RISK_FACTOR_Family history unknown for breast cancer,RISK_FACTOR_History of ovarian cancer,RISK_FACTOR_History of endometrial cancer,RISK_FACTOR_Not available,BIRTH_DATE,GENDER_TITLE,RACE_KOREAN,RACE_WHITE,RACE_INDIAN (ASIAN),RACE_BLACK,RACE_UNREPORTED/CHOSE NOT TO DISCLOSE,RACE_OTHER ASIAN,RACE_VIETNAMESE,RACE_CHINESE,RACE_AMERICAN INDIAN/ALASKA NATIVE,RACE_FILIPINO,RACE_OTHER,RACE_JAPANESE,RACE_HAWAIIAN,RACE_OTHER PACIFIC ISLANDER,ETHNIC_NOT HISPANIC OR LATINO,"ETHNIC_ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",ETHNIC_UNREPORTED/CHOSE NOT TO DISCLOSE,ETHNIC_HISPANIC OR LATINO,"ETHNIC_MEXICAN, MEXICAN AMERICAN, CHICANO",ETHNIC_PUERTO RICAN,race
0,4330421371,45,65330676,2018-12-03,2018-12-03,SCREEN,INDEX-1,2018-10-15,1968.0,OZ,64.0,IN,21.11,11,26,0,0,0,0,2.0,3.0,2018-11-10,PRE,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1973-07-01,FEMALE,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,ASIAN
1,4330434822,40,77664732,2018-08-16,2018-08-16,SCREEN,INDEX-1,2018-10-03,1606.4,OZ,60.0,IN,19.61,14,23,0,0,0,0,3.0,3.0,NaN,NaN,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1978-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,WHITE
2,4330449572,69,63998027,2019-08-24,2019-08-24,SCREEN,INDEX-1,2019-10-29,2320.0,OZ,61.5,IN,26.95,14,32,52,0,0,0,1.0,1.0,NaN,NaN,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1950-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,WHITE
3,4330495728,56,77791327,2018-09-11,2018-09-11,SCREEN,INDEX-1,2018-07-18,2224.0,OZ,62.5,IN,25.02,13,36,48,0,0,0,2.0,2.0,NaN,POSTNAT,1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1962-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,WHITE
4,4333002438,56,67560128,2020-12-12,2020-12-12,SCREEN,INDEX-1,2021-11-24,2096.0,OZ,66.5,IN,20.83,13,31,49,0,0,0,3.0,3.0,NaN,POSTNAT,0,0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,1964-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,WHITE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250,4335822268,68,73206127,2016-12-21,2016-12-21,SCREEN,INDEX-1,2016-11-22,2281.6,OZ,NaN,IN,22.67,12,0,55,0,0,0,0.0,0.0,NaN,POSTNAT,0,1,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,1948-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,WHITE
251,4335849525,69,67050532,2021-04-23,2021-04-23,SCREEN,INDEX-1,2021-05-20,2800.0,OZ,66.0,IN,28.25,11,25,49,49,0,0,3.0,3.0,NaN,POSTNAT,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1951-07-01,FEMALE,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,WHITE
252,4337018091,67,60768510,2020-05-27,2020-05-27,SCREEN,INDEX-1,2020-05-26,2400.0,OZ,63.5,IN,26.15,12,27,0,0,0,0,3.0,3.0,NaN,POSTNAT,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1952-07-01,FEMALE,0,1,0,0,0,0,0,

In [35]:
def race_summary(df, label):
    n_total = len(df)
    counts = df['race'].value_counts(dropna=False)
    tbl = pd.DataFrame({
        'Race':  counts.index,
        'Count': counts.values,
        '%':     (counts.values / n_total * 100).round(1),
    })
    print(f"{'─'*40}")
    print(f"{label}  (n={n_total:,})")
    print(f"{'─'*40}")
    print(tbl.to_string(index=False))
    print()

race_summary(mo_cancer_ehr,        'MO Cancer')
race_summary(normal_candidate_ehr, 'Normal Candidate')

────────────────────────────────────────
MO Cancer  (n=255)
────────────────────────────────────────
         Race  Count    %
        WHITE    235 92.2
        BLACK     11  4.3
        ASIAN      6  2.4
OTHER/UNKNOWN      3  1.2

────────────────────────────────────────
Normal Candidate  (n=31,060)
────────────────────────────────────────
                         Race  Count    %
                        WHITE  27976 90.1
                        BLACK   1502  4.8
                OTHER/UNKNOWN    791  2.5
                        ASIAN    755  2.4
AMERICAN INDIAN/ALASKA NATIVE     23  0.1
    HAWAIIAN/PACIFIC ISLANDER     13  0.0



In [40]:
mo_cancer_ehr['PATIENT_STUDY_ID'].nunique(), normal_candidate_ehr['PATIENT_STUDY_ID'].nunique()

(253, 30903)

In [41]:
# ── Find duplicate PATIENT_STUDY_IDs in EHR tables ───────────────────────────

def find_duplicates(df, label):
    dups = df[df.duplicated(subset='PATIENT_STUDY_ID', keep=False)].copy()
    dup_ids = dups['PATIENT_STUDY_ID'].unique()
    print(f"{'─'*60}")
    print(f"{label}: {len(dup_ids)} patient(s) with duplicate rows")
    print(f"{'─'*60}")
    for pid in dup_ids:
        rows = df[df['PATIENT_STUDY_ID'] == pid]
        print(f"\nPATIENT_STUDY_ID: {pid}  ({len(rows)} rows)")
        # Show columns that differ between the duplicate rows
        varying = [c for c in rows.columns if rows[c].nunique(dropna=False) > 1]
        if varying:
            print(f"  Columns that differ: {varying}")
            print(rows[['PATIENT_STUDY_ID'] + varying].to_string(index=True))
        else:
            print("  All columns identical (exact duplicate rows)")
    print()

find_duplicates(mo_cancer_ehr,        'MO Cancer EHR')
find_duplicates(normal_candidate_ehr, 'Normal Candidate EHR')

────────────────────────────────────────────────────────────
MO Cancer EHR: 2 patient(s) with duplicate rows
────────────────────────────────────────────────────────────

PATIENT_STUDY_ID: 4333004761  (2 rows)
  Columns that differ: ['PatientAge', 'StudyDate']
   PATIENT_STUDY_ID  PatientAge   StudyDate
7        4333004761          44  2021-06-26
8        4333004761          45  2021-07-09

PATIENT_STUDY_ID: 4334479084  (2 rows)
  Columns that differ: ['StudyDate']
     PATIENT_STUDY_ID   StudyDate
219        4334479084  2021-12-14
220        4334479084  2021-12-21

────────────────────────────────────────────────────────────
Normal Candidate EHR: 153 patient(s) with duplicate rows
────────────────────────────────────────────────────────────

PATIENT_STUDY_ID: 4330448643  (2 rows)
  Columns that differ: ['ACCESSION_NUMBER']
     PATIENT_STUDY_ID  ACCESSION_NUMBER
304        4330448643         455725861
305        4330448643         455790436

PATIENT_STUDY_ID: 4330461945  (2 rows)
  Co

In [ ]:


# ── STEP 3: Age bin (5-year) ──────────────────────────────────────────────────
AGE_BINS   = list(range(30, 100, 5))
AGE_LABELS = [f'{b}-{b+4}' for b in AGE_BINS[:-1]]

mo_cancer_ehr['age_bin']        = pd.cut(mo_cancer_ehr['PatientAge'],        bins=AGE_BINS, labels=AGE_LABELS, right=True)
normal_candidate_ehr['age_bin'] = pd.cut(normal_candidate_ehr['PatientAge'], bins=AGE_BINS, labels=AGE_LABELS, right=True)

# ── STEP 4: Feature tables ────────────────────────────────────────────────────
c_feat = c_combo.merge(mo_cancer_ehr[['PATIENT_STUDY_ID', 'PatientAge', 'age_bin', 'race']], on='PATIENT_STUDY_ID', how='left')
n_feat = n_combo.merge(normal_candidate_ehr[['PATIENT_STUDY_ID', 'PatientAge', 'age_bin', 'race']], on='PATIENT_STUDY_ID', how='left')

def make_stratum(df):
    return df['birads_combo'].astype(str) + '|' + df['race'].astype(str) + '|' + df['age_bin'].astype(str)

c_feat['stratum'] = make_stratum(c_feat)
n_feat['stratum'] = make_stratum(n_feat)

# ── STEP 5: Match ─────────────────────────────────────────────────────────────
rng = np.random.default_rng(RANDOM_SEED)
normal_pool = n_feat.copy()
normal_pool_by_stratum = {s: grp.index.tolist() for s, grp in normal_pool.groupby('stratum')}

matched_cancer_rows, matched_normal_rows, unmatched_cancer_ids = [], [], []
used_normal_indices = set()

for _, cancer_row in c_feat.iterrows():
    stratum   = cancer_row['stratum']
    available = [i for i in normal_pool_by_stratum.get(stratum, []) if i not in used_normal_indices]

    # Fallback: relax age bin ±5 then ±10 years if stratum is empty
    if len(available) == 0:
        birads, race, age_bin = stratum.split('|')
        age_idx = AGE_LABELS.index(age_bin) if age_bin in AGE_LABELS else -1
        for offset in [-1, 1, -2, 2]:
            adj_idx = age_idx + offset
            if 0 <= adj_idx < len(AGE_LABELS):
                adj_stratum = f'{birads}|{race}|{AGE_LABELS[adj_idx]}'
                available += [i for i in normal_pool_by_stratum.get(adj_stratum, []) if i not in used_normal_indices]

    if len(available) == 0:
        unmatched_cancer_ids.append(cancer_row['PATIENT_STUDY_ID'])
        continue

    chosen_idx = rng.choice(available, size=min(N_MATCH, len(available)), replace=False).tolist()
    used_normal_indices.update(chosen_idx)
    matched_cancer_rows.append(cancer_row)
    for idx in chosen_idx:
        row = normal_pool.loc[idx].copy()
        row['matched_cancer_id'] = cancer_row['PATIENT_STUDY_ID']
        matched_normal_rows.append(row)

matched_cancer  = pd.DataFrame(matched_cancer_rows).reset_index(drop=True)
matched_normals = pd.DataFrame(matched_normal_rows).reset_index(drop=True)

# ── STEP 6: Summary ───────────────────────────────────────────────────────────
print(f"Cancer matched    : {len(matched_cancer)} / {len(c_feat)}")
print(f"Normals selected  : {len(matched_normals)}  (ratio {len(matched_normals)/len(matched_cancer):.2f}:1)")
print(f"Unmatched IDs     : {unmatched_cancer_ids}")

In [ ]:


# ── STEP 3: Age bin (5-year) ──────────────────────────────────────────────────
AGE_BINS   = list(range(30, 100, 5))
AGE_LABELS = [f'{b}-{b+4}' for b in AGE_BINS[:-1]]

mo_cancer_ehr['age_bin']        = pd.cut(mo_cancer_ehr['PatientAge'],        bins=AGE_BINS, labels=AGE_LABELS, right=True)
normal_candidate_ehr['age_bin'] = pd.cut(normal_candidate_ehr['PatientAge'], bins=AGE_BINS, labels=AGE_LABELS, right=True)

# ── STEP 4: Feature tables ────────────────────────────────────────────────────
c_feat = c_combo.merge(mo_cancer_ehr[['PATIENT_STUDY_ID', 'PatientAge', 'age_bin', 'race']], on='PATIENT_STUDY_ID', how='left')
n_feat = n_combo.merge(normal_candidate_ehr[['PATIENT_STUDY_ID', 'PatientAge', 'age_bin', 'race']], on='PATIENT_STUDY_ID', how='left')

def make_stratum(df):
    return df['birads_combo'].astype(str) + '|' + df['race'].astype(str) + '|' + df['age_bin'].astype(str)

c_feat['stratum'] = make_stratum(c_feat)
n_feat['stratum'] = make_stratum(n_feat)

# ── STEP 5: Match ─────────────────────────────────────────────────────────────
rng = np.random.default_rng(RANDOM_SEED)
normal_pool = n_feat.copy()
normal_pool_by_stratum = {s: grp.index.tolist() for s, grp in normal_pool.groupby('stratum')}

matched_cancer_rows, matched_normal_rows, unmatched_cancer_ids = [], [], []
used_normal_indices = set()

for _, cancer_row in c_feat.iterrows():
    stratum   = cancer_row['stratum']
    available = [i for i in normal_pool_by_stratum.get(stratum, []) if i not in used_normal_indices]

    # Fallback: relax age bin ±5 then ±10 years if stratum is empty
    if len(available) == 0:
        birads, race, age_bin = stratum.split('|')
        age_idx = AGE_LABELS.index(age_bin) if age_bin in AGE_LABELS else -1
        for offset in [-1, 1, -2, 2]:
            adj_idx = age_idx + offset
            if 0 <= adj_idx < len(AGE_LABELS):
                adj_stratum = f'{birads}|{race}|{AGE_LABELS[adj_idx]}'
                available += [i for i in normal_pool_by_stratum.get(adj_stratum, []) if i not in used_normal_indices]

    if len(available) == 0:
        unmatched_cancer_ids.append(cancer_row['PATIENT_STUDY_ID'])
        continue

    chosen_idx = rng.choice(available, size=min(N_MATCH, len(available)), replace=False).tolist()
    used_normal_indices.update(chosen_idx)
    matched_cancer_rows.append(cancer_row)
    for idx in chosen_idx:
        row = normal_pool.loc[idx].copy()
        row['matched_cancer_id'] = cancer_row['PATIENT_STUDY_ID']
        matched_normal_rows.append(row)

matched_cancer  = pd.DataFrame(matched_cancer_rows).reset_index(drop=True)
matched_normals = pd.DataFrame(matched_normal_rows).reset_index(drop=True)

# ── STEP 6: Summary ───────────────────────────────────────────────────────────
print(f"Cancer matched    : {len(matched_cancer)} / {len(c_feat)}")
print(f"Normals selected  : {len(matched_normals)}  (ratio {len(matched_normals)/len(matched_cancer):.2f}:1)")
print(f"Unmatched IDs     : {unmatched_cancer_ids}")